In [ ]:
--18号10:51之后，预路由输出的recommend_member_risk_price=0.2的客户，且经过商业化的规则后，会曝光20的会员
--26号扩充灰度到50%，其中50%用户可见，可给20，另外50%用户不可见，不会输出20

SELECT  decisionid
       ,requestid
       ,decision_time
       ,engineversion                 --engineversion>=36是灰度扩大到50%
       ,get_json_object(input,"$.user_no") user_no
       ,get_json_object(output,"$.recommend_member_risk_price") recommend_member_risk_price
       ,ROW_NUMBER() OVER (PARTITION BY get_json_object(input,"$.user_no"),to_date(decision_time) ORDER BY  decision_time DESC) rn_desc
FROM xyf_dwd.dwd_inloan_t_decision_result_detail_df rd
WHERE 1 = 1
AND pt = max_pt("xyf_dwd.dwd_inloan_t_decision_result_detail_df")
AND enginecode = "jcl_20251125000001"
AND to_date(decision_time) >= "2026-05-26"


-- 业务引擎取实时额度  --会员卡签约的时候会调用
SELECT  *
       ,GET_JSON_OBJECT(input_data,"$.available_amt_and_freeze_amt")      AS 可用额度
FROM xyf_dwd.dwd_xf_business_engine_sync_model_log_di
WHERE pt = '20260531'
AND model_code = 'vip_crowd_predict'
AND input_data LIKE '%vip_pricing_crowd%'
AND GET_JSON_OBJECT(input_data, '$.subVipType') = "fei_yue_i20"
LIMIT 100


-- 老客当天实时可用额度
SELECT  cust_no,available_amt_new
FROM xyf_dwd.dwd_preloan_account_amount_change_df
WHERE product_code IN ('personal_loan') --取个人分期产品的数据 
AND pt = MAX_PT("xyf_dwd.dwd_preloan_account_amount_change_df")
AND biz_type in('ADJUST_LIMIT', 'TEMPORARY_TAKE_EFFECT') --固额调额、临额生效 
AND date(occurrence_time) >= '2024-01-01'


In [ ]:
DROP TABLE IF EXISTS xyf_jingying_dev.price20_preroute_cust_0519_lss;
CREATE TABLE xyf_jingying_dev.price20_preroute_cust_0519_lss AS

SELECT  m.* 
FROM
(
SELECT  t.pt                                                                                             AS 曝光日期
       ,t.user_no
       ,t.cust_no
       ,t.zijin_decision_time
       ,t.埋点时间
       ,t.授信更新时间
       ,t.first_order_time
       ,t.loan_time
	   ,t.order_number
       ,t.first_order_number
       ,t.order_amt
       ,t.risk_status
       ,t.loan_status
       ,t.loan_amt
       ,t.period
       ,t.asset_type_flag
       ,t.fee_rate
       ,CASE 
			WHEN t.pt < '2026-05-27' THEN 
				CASE 
					WHEN xyf_dwd.randomv3('jkjzzplk', t.user_no, 3) BETWEEN 0 AND 99 THEN '测试组'
					WHEN xyf_dwd.randomv3('jkjzzplk', t.user_no, 3) BETWEEN 100 AND 999 THEN '对照组'
					ELSE '其他'
				END
			WHEN t.pt >= '2026-05-27' THEN 
				CASE 
					WHEN xyf_dwd.randomv3('jkjzzplk', t.user_no, 3) BETWEEN 0 AND 499 THEN '测试组'
					WHEN xyf_dwd.randomv3('jkjzzplk', t.user_no, 3) BETWEEN 500 AND 999 THEN '对照组'
					ELSE '其他'
				END
			ELSE '其他'
		END AS group_tag  -- 0526 18:07:12 改到50%
       ,t.风险原始定价
       ,t.飞跃会员类型分组
       ,t.vip_classify_group
       ,t.recommend_member_risk_price
       ,CASE WHEN t.recommend_member_risk_price = 0.2 THEN 1  ELSE 0 END                                 AS risk_I20
	   ,CASE WHEN t.debit_amt >= 4000 AND t.debit_amt < 10000 THEN 1  ELSE 0 END                         AS 可用额度4k_10k
       ,CASE WHEN t.debit_amt < 4000 THEN '[0k,4k)'
             WHEN t.debit_amt < 5000 THEN '[4k,5k)'
             WHEN t.debit_amt < 6000 THEN '[5k,6k)'
             WHEN t.debit_amt < 7000 THEN '[6k,7k)'
             WHEN t.debit_amt < 8000 THEN '[7k,8k)'
             WHEN t.debit_amt < 9000 THEN '[8k,9k)'
             WHEN t.debit_amt < 10000 THEN '[9k,10k)'
             WHEN t.debit_amt >= 10000 THEN '[10k,+)'  ELSE '其他' END                                     AS 额度区间
       
FROM
(
	SELECT  t2.*
	       ,lo.*except(user_no,cust_no)
	FROM
	(
		SELECT  t1.*
		       ,available_amt.occurrence_time AS 授信更新时间
		       ,available_amt.debit_amt
		FROM
		(
			SELECT  m.pt
			       ,m.user_no
			       ,c.cust_no
			       ,m.埋点时间
			       ,m.风险原始定价
			       ,m.飞跃会员类型分组
			       ,m.vip_classify_group
			       ,zijin.decision_time AS zijin_decision_time
			       ,zijin.recommend_member_risk_price
			FROM
			(
				SELECT  user_no
				       ,DATE(trigger_timestamp)                                                                  AS pt
				       ,trigger_timestamp                                                                        AS 埋点时间
				       ,GET_JSON_OBJECT(biz_info,"$.sub_vip_type")                                               AS 飞跃会员类型分组
				       ,risk_price                                                                               AS 风险原始定价  --与周报看板上的逻辑是一样的
				       ,vip_classify_group
				FROM xyf_dwd.dwd_event_tracking_log_di
				WHERE pt >= '20260519'
				AND DATE(TO_DATE(pt, "yyyyMMdd")) = DATE(trigger_timestamp)
				AND tracking_id = 'XYF_H5_003082'      --借款页曝光、走什么会员卡决策流、什么定价都已经决定好了 
				AND user_no NOT IN ("1061112123", "1055063706", "1043199921", "1028160229", "1034141205") 
				-- AND vip_classify_group = 'fy_group_2' --仅筛选飞跃转化流可见用户
				-- QUALIFY ROW_NUMBER() OVER(PARTITION BY user_no , date(trigger_timestamp) ORDER BY trigger_timestamp ASC ) = 1 ---每日首曝口径
			) m
			LEFT JOIN
			(
				SELECT  cust_no
				       ,app_user_id
				FROM xyf_dim.dim_user_app_basic_info_df
				WHERE pt = MAX_PT('xyf_dim.dim_user_app_basic_info_df')
				AND app IN ('xyf01', 'fxk') 
			) c
			ON m.user_no = c.app_user_id
			LEFT JOIN
			(
				SELECT  decisionid
				       ,requestid
				       ,decision_time
				       ,engineversion         --engineversion >= 36是灰度扩大到50%
				       ,GET_JSON_OBJECT(input,"$.user_no")                                      AS user_no
				       ,CAST(GET_JSON_OBJECT(output,"$.recommend_member_risk_price") AS DOUBLE) AS recommend_member_risk_price
					   --, ROW_NUMBER() OVER (PARTITION BY get_json_object(input, "$.user_no"), to_date(decision_time) ORDER BY  decision_time DESC) rn_desc
				FROM xyf_dwd.dwd_inloan_t_decision_result_detail_df 
				WHERE 1 = 1
				AND pt = MAX_PT("xyf_dwd.dwd_inloan_t_decision_result_detail_df")
				AND enginecode = "jcl_20251125000001"
				AND TO_DATE(decision_time) >= "2026-05-19" 
			) zijin
			ON m.user_no = zijin.user_no AND zijin.decision_time <= m.埋点时间 
			QUALIFY ROW_NUMBER() OVER ( PARTITION BY m.user_no, m.埋点时间 ORDER BY zijin.decision_time DESC NULLS LAST ) = 1 --优先取有时间的最近一条；实在没有匹配记录，才保留那条空的 LEFT JOIN 结果
			-- 取埋点时间前最近一条资金决策记录
		) t1


		
		LEFT JOIN
		(
			SELECT  cust_no
			       ,occurrence_time
			       ,available_amt_new AS debit_amt
			FROM xyf_dwd.dwd_preloan_account_amount_change_df
			WHERE product_code IN ('personal_loan')           --取个人分期产品的数据
			AND pt = MAX_PT("xyf_dwd.dwd_preloan_account_amount_change_df")
			AND biz_type IN ('OPEN_ACCOUNT', 'ADJUST_LIMIT', 'TEMPORARY_TAKE_EFFECT')    --固额调额、临额生效、初始额度 
			-- AND DATE(occurrence_time) >= '2026-01-01' 
		) available_amt
		ON t1.cust_no = available_amt.cust_no AND available_amt.occurrence_time <= t1.埋点时间 
		QUALIFY ROW_NUMBER() OVER ( PARTITION BY t1.user_no, t1.埋点时间 ORDER BY available_amt.occurrence_time DESC NULLS LAST ) = 1
		 -- 取埋点时间前最近一条老客额度记录
	) t2
	LEFT JOIN
	(
		SELECT  order_number
		       ,user_no
		       ,cust_no
		       ,first_order_number
		       ,first_order_time
		       ,order_amt
		       ,risk_status
		       ,loan_status
		       ,loan_time
		       ,loan_amt
		       ,period
		       ,asset_type_flag
		       ,fee_rate
		FROM xyf_dws.dws_inloan_user_order_df
		WHERE pt = MAX_PT('xyf_dws.dws_inloan_user_order_df')
		AND app IN ('xyf01')
		AND business_line IN ('APP', '小程序端')
		AND DATE(first_order_time) >= '2026-01-22'
		AND loan_flag <> '首贷' 
	) lo
	ON t2.user_no = lo.user_no AND lo.first_order_time >= t2.埋点时间  AND DATE(lo.first_order_time) = DATE(t2.埋点时间)
	QUALIFY ROW_NUMBER() OVER ( PARTITION BY t2.user_no, t2.埋点时间 ORDER BY lo.first_order_time ASC NULLS LAST ) = 1 
	--取埋点时间后最近一条订单记录
) t
QUALIFY ROW_NUMBER() OVER ( PARTITION BY t.user_no, t.order_number, t.pt ORDER BY t.埋点时间 DESC NULLS LAST ) = 1
-- 用户可能当日一笔订单前有多条曝光记录，导致重复，最终筛选逻辑
-- 有订单：按 user_no + order_number + 曝光日期 去重，每个用户每天每笔订单保留一条最新曝光。
-- 没订单：按 user_no + NULL + 曝光日期 去重，每个用户每天无订单曝光保留一条最新曝光。
) m
LEFT JOIN
(
	SELECT  user_no
	       ,cust_no
	       ,DATE(loan_time) AS 首贷时间
	       ,loan_amt
	FROM xyf_dws.dws_inloan_user_order_df
	WHERE pt = MAX_PT('xyf_dws.dws_inloan_user_order_df')
	AND app IN ('xyf01', 'fxk')
	AND loan_status = 'success'
	AND loan_flag = '首贷' 
) shoudai
ON shoudai.cust_no = m.cust_no AND m.曝光日期 > shoudai.首贷时间
WHERE shoudai.user_no IS NOT NULL --筛选复贷用户 


In [ ]:
DROP TABLE IF EXISTS xyf_jingying_dev.price20_preroute_cust_0519_lss_info;
CREATE TABLE xyf_jingying_dev.price20_preroute_cust_0519_lss_info AS
SELECT  m.曝光日期
       ,m.user_no
       ,m.cust_no
       ,m.zijin_decision_time
       ,m.埋点时间
       ,m.授信更新时间
       ,m.first_order_time
       ,r.risk_success_time
       ,m.loan_time
      --会员卡信息 
       ,fy.sub_vip_type                                                    AS fy_sub_vip_type
       ,fy.vip_order_number                                                AS fy_vip_order_number
       ,CASE WHEN fy.app_user_id IS NULL THEN '非在会'  ELSE '在会' END     AS 飞跃在会 
       --,CASE WHEN fy.renew_period = 0 AND m.first_order_number = fy.loan_order_number THEN '签约当笔'
       --      WHEN fy.renew_period = 0 AND m.first_order_number <> fy.loan_order_number THEN '签约期间发起'
       --      WHEN fy.renew_period > 0 THEN '续约期间发起'  ELSE NULL END    AS 飞跃在会状态
       ,CASE WHEN fy.loan_order_number = m.first_order_number THEN '签约当笔'
             WHEN fy.loan_order_number <> m.first_order_number THEN '签约期间发起' ELSE NULL END  AS 飞跃在会状态
       ,fy.start_time                                                      AS 飞跃会员卡开始时间
       ,fy.end_time                                                        AS 飞跃会员卡结束时间
       ,fy.pay_time                                                        AS fy_pay_time 
       ,fy.real_card_price                                                 AS fy_real_card_price
       ,fy.act_refund_time                                                 AS fy_act_refund_time
       ,fy.refund_amount                                                   AS fy_refund_amount
       
       ,fx.vip_order_number                                                AS fx_vip_order_number
       ,CASE WHEN fx.app_user_id IS NULL THEN '非在会' ELSE '在会' END      AS 飞享在会
       ,CASE WHEN fx.loan_order_number = m.first_order_number THEN '签约当笔'
             WHEN fx.loan_order_number <> m.first_order_number THEN '签约期间发起' ELSE NULL END  AS 飞享在会状态
       ,fx.start_time                                                      AS 飞享会员卡开始时间
       ,fx.end_time                                                        AS 飞享会员卡结束时间
       ,fx.pay_time                                                        AS fx_pay_time
       ,fx.real_card_price /100                                            AS fx_real_card_price
       ,fx.act_refund_time                                                 AS fx_act_refund_time
       ,fx.refund_amount  /100                                             AS fx_refund_amount

       ,tek.pay_time                                                       AS tek_pay_time
       ,tek.pay_amt                                                        AS tek_pay_amt
       ,tek.refund_amt                                                     AS tek_refund_amt
       ,tek.tek_cnt                                                        AS tek_cnt
       --订单信息 
       ,m.order_number
       ,m.first_order_number
       ,m.order_amt
       ,m.risk_status
       ,m.loan_status
       ,m.loan_amt
       ,m.period
       ,m.asset_type_flag
       ,m.fee_rate
       ,m.风险原始定价
       --分组信息 
       ,m.group_tag
       ,m.飞跃会员类型分组
       ,m.vip_classify_group
       ,m.recommend_member_risk_price
       ,m.risk_I20
       ,m.可用额度4k_10k
       ,m.额度区间
FROM xyf_jingying_dev.price20_preroute_cust_0519_lss m
-- 风险通过时间 
LEFT JOIN
(
	SELECT  DISTINCT ori_order_number
	       ,risk_success_time --去重，只留一条包含风险通过时间的记录 
	FROM xyf_dwd.dwd_inloan_loan_apply_hf
	WHERE pt = MAX_PT('xyf_dwd.dwd_inloan_loan_apply_hf')
	AND DATE(main_date_created) >= '2026-01-01' -- 预过滤 
 
) r
ON m.first_order_number = r.ori_order_number
-- 飞跃在会状态 （新逻辑：基于会员卡有效期） 
LEFT JOIN
(
	SELECT  app_user_id
	       ,vip_order_number --, first_vip_order_number 
	       ,order_time
	       ,pay_time
	       ,real_card_price
	       ,act_refund_time
	       ,refund_amount
	       ,start_time
	       ,coalesce(CAST(failure_time AS DATETIME),end_time) AS end_time --飞跃合约期外退款，failure_time = end_time 
	       ,loan_order_number
	       ,vip_status
	       ,renew_period
	       ,vip_flow_group --灰度组 
	       ,sub_vip_type --子会员类型 
	FROM xyf_dwd.dwd_inloan_leap_vip_order_hf
	WHERE pt = MAX_PT('xyf_dwd.dwd_inloan_leap_vip_order_hf')
	AND DATE(order_time) >= '2026-01-01' 
) fy
ON m.user_no = fy.app_user_id AND m.曝光日期 >= DATE(fy.start_time) AND m.曝光日期 <= DATE(fy.end_time)
-- 飞享在会状态 
LEFT JOIN
(
	SELECT  app_user_id
	       ,vip_order_number
	       ,first_vip_order_number
	       ,order_time
	       ,pay_time
              ,real_card_price
	       ,act_refund_time
	       ,refund_amount
	       ,start_time
	       ,LEAST(CAST(failure_time AS DATETIME),end_time) AS end_time --飞享存在合约期外退款，把refund_time作为failure_time，因此取两者较小值 
	       ,order_number_loan                              AS loan_order_number
	FROM xyf_dwd.dwd_user_vip_order_df
	WHERE pt = MAX_PT('xyf_dwd.dwd_user_vip_order_df')
	AND vip_card_type = 1
	AND if_validation <> 0
	AND DATE(order_time) >= '2026-01-01' 
) fx
ON m.user_no = fx.app_user_id AND m.曝光日期 >= DATE(fx.start_time) AND m.曝光日期 <= DATE(fx.end_time)
LEFT JOIN
(
       SELECT  app_user_id
              ,loan_order_number
              ,COUNT(DISTINCT order_no)                                                        AS tek_cnt
              ,MAX(order_time)                                                                 AS pay_time
              ,SUM(real_order_price)                                                           AS pay_amt
              ,SUM(CASE WHEN act_refund_time IS NOT NULL THEN NVL(refund_amount,0) ELSE 0 END) AS refund_amt
       FROM xyf_dwd.dwd_user_tek_order_df
       WHERE pt = MAX_PT('xyf_dwd.dwd_user_tek_order_df')
       GROUP BY  app_user_id
                ,loan_order_number
) tek
ON m.user_no = tek.app_user_id AND tek.loan_order_number = m.first_order_number


QUALIFY ROW_NUMBER() OVER (
    PARTITION BY user_no,埋点时间
    ORDER BY COALESCE(飞跃会员卡开始时间, '9999-12-31') ASC,
             COALESCE(飞享会员卡开始时间, '9999-12-31') ASC
) = 1     --去重，因续约订单开始日期与上一笔订单结束日相同，清除重复订单，保留飞跃和飞享会员卡开始时间最早的记录
;


#### 窗口期转化口径

In [1]:
query = '''
WITH exposure_base AS
(
       SELECT  user_no
              ,曝光日期
              ,group_tag
              ,MAX(可用额度4k_10k) AS 可用额度4k_10k
              ,MAX(risk_I20)   AS risk_I20
       FROM xyf_jingying.price20_preroute_cust_0519_lss_info
       WHERE 1 = 1
       GROUP BY  user_no
              ,曝光日期
              ,group_tag
),
-- 放款息费、定价、期限
 loan_order_detail AS
(
	SELECT  m.user_no
	       ,m.曝光日期
	       ,m.group_tag
	       ,m.risk_I20
	       ,m.可用额度4k_10k
	       ,DATEDIFF(DATE(lo.first_order_time),m.曝光日期) AS 发起距首曝天数
	       ,lo.* EXCEPT(user_no)
	FROM exposure_base m
	LEFT JOIN
	(
		SELECT  order_number
		       ,user_no
		       ,first_order_number
		       ,first_order_time
		       ,order_amt
		       ,risk_status
		       ,loan_status
		       ,loan_time
		       ,loan_amt
		       ,period
		       ,asset_type_flag
		       ,fee_rate
		FROM xyf_dws.dws_inloan_user_order_df
		WHERE pt = MAX_PT('xyf_dws.dws_inloan_user_order_df')
		AND app IN ('xyf01')
		AND business_line IN ('APP', '小程序端')
		AND DATE(first_order_time) >= '2026-05-01'
		AND loan_flag <> '首贷'
		AND loan_status = 'success' 
	) lo
	ON m.user_no = lo.user_no AND DATEDIFF(DATE(lo.first_order_time), m.曝光日期) BETWEEN 0 AND 30
), repay_plan AS
(
	SELECT  order_number
	       ,SUM(initial_principal)                                                          AS principal
	       ,SUM(initial_interest) + SUM(initial_after_loan_fee) + SUM(initial_platform_fee) AS interest_fee
	FROM xyf_dwd.dwd_repay_loan_repay_plan_df
	WHERE pt = MAX_PT('xyf_dwd.dwd_repay_loan_repay_plan_df')
	GROUP BY  order_number
), order_level AS
(
	SELECT  a.*
	       ,b.principal
	       ,b.interest_fee
	FROM loan_order_detail a
	LEFT JOIN repay_plan b
	ON a.order_number = b.order_number
), loan_agg AS
(
SELECT  曝光日期
       ,group_tag
       ,risk_I20
       ,可用额度4k_10k
       ,COUNT(DISTINCT user_no)                                                           AS 曝光人数

       ,SUM(CASE WHEN 发起距首曝天数 = 0 THEN loan_amt ELSE 0 END)                         AS 放款金额_T0
       ,COUNT(DISTINCT CASE WHEN 发起距首曝天数 = 0 THEN order_number END)                 AS 放款笔数_T0
       ,SUM(CASE WHEN 发起距首曝天数 = 0 THEN loan_amt * fee_rate ELSE 0 END)              AS 定价_T0
       ,SUM(CASE WHEN 发起距首曝天数 = 0 THEN loan_amt * period ELSE 0 END)                AS 期限_T0
       ,SUM(CASE WHEN 发起距首曝天数 = 0 THEN principal ELSE 0 END)                        AS principal_T0
       ,SUM(CASE WHEN 发起距首曝天数 = 0 THEN interest_fee ELSE 0 END)                     AS annualized_interest_fee_T0

       ,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 6 THEN loan_amt ELSE 0 END)             AS 放款金额_T7
       ,COUNT(DISTINCT CASE WHEN 发起距首曝天数 BETWEEN 0 AND 6 THEN order_number END)     AS 放款笔数_T7
       ,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 6 THEN loan_amt * fee_rate ELSE 0 END)  AS 定价_T7
       ,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 6 THEN loan_amt * period ELSE 0 END)    AS 期限_T7
       ,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 6 THEN principal ELSE 0 END)            AS principal_T7
       ,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 6 THEN interest_fee ELSE 0 END)         AS annualized_interest_fee_T7

       ,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 30 THEN loan_amt ELSE 0 END)            AS 放款金额_T30
       ,COUNT(DISTINCT CASE WHEN 发起距首曝天数 BETWEEN 0 AND 30 THEN order_number END)    AS 放款笔数_T30
       ,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 30 THEN loan_amt * fee_rate ELSE 0 END) AS 定价_T30
       ,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 30 THEN loan_amt * period ELSE 0 END)   AS 期限_T30
       ,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 30 THEN principal ELSE 0 END)           AS principal_T30
       ,SUM(CASE WHEN 发起距首曝天数 BETWEEN 0 AND 30 THEN interest_fee ELSE 0 END)        AS annualized_interest_fee_T30
FROM order_level
GROUP BY  曝光日期
         ,group_tag
         ,risk_I20
         ,可用额度4k_10k
),

--会员卡收入
vip_order_raw AS
(
	SELECT  app_user_id
	       ,cust_no
	       ,order_time
	       ,'飞跃'                                                                      AS card_type
	       ,real_card_price                                                             AS sign_amt
	       ,CASE WHEN pay_time IS NOT NULL THEN real_card_price  ELSE 0 END             AS pay_amt
	       ,CASE WHEN act_refund_time IS NOT NULL THEN NVL(refund_amount,0)  ELSE 0 END AS refund_amt
	FROM xyf_dwd.dwd_inloan_leap_vip_order_hf
	WHERE pt = MAX_PT('xyf_dwd.dwd_inloan_leap_vip_order_hf')
	AND DATE(order_time) >= '2025-05-24' 

	UNION ALL
	SELECT  app_user_id
	       ,cust_no
	       ,order_time
	       ,'飞享'                                                                            AS card_type
	       ,real_card_price / 100                                                             AS sign_amt
	       ,CASE WHEN pay_time IS NOT NULL THEN real_card_price / 100  ELSE 0 END             AS pay_amt
	       ,CASE WHEN act_refund_time IS NOT NULL THEN NVL(refund_amount,0) / 100  ELSE 0 END AS refund_amt
	FROM xyf_dwd.dwd_user_vip_order_df
	WHERE pt = MAX_PT('xyf_dwd.dwd_user_vip_order_df')
	AND vip_card_type = 1
	AND if_validation <> 0 
	
	UNION ALL
	SELECT  app_user_id
	       ,cust_no
	       ,order_time
	       ,'提额'                                                                      AS card_type
	       ,real_order_price                                                            AS sign_amt
	       ,real_order_price                                                            AS pay_amt
	       ,CASE WHEN act_refund_time IS NOT NULL THEN NVL(refund_amount,0)  ELSE 0 END AS refund_amt
	FROM xyf_dwd.dwd_user_tek_order_df
	WHERE pt = MAX_PT('xyf_dwd.dwd_user_tek_order_df') 
), vip_order_detail AS
(
	SELECT  m.曝光日期
	       ,m.user_no
	       ,m.group_tag
	       ,m.risk_I20
	       ,m.可用额度4k_10k
	       ,DATEDIFF(DATE(v.order_time),m.曝光日期) AS 会员卡距首曝天数
	       ,v.card_type
	       ,v.sign_amt
	       ,v.pay_amt - v.refund_amt            AS equity_income
	FROM exposure_base m
	LEFT JOIN vip_order_raw v
	ON m.user_no = v.app_user_id 
    AND DATEDIFF(DATE(v.order_time), m.曝光日期) BETWEEN 0 AND 30
), vip_agg AS
(
SELECT  曝光日期
       --,风险原始定价
       --,vip_classify_group
       --,飞跃会员类型分组
       ,group_tag
       ,risk_I20
       --,额度区间
       ,可用额度4k_10k
       ,SUM(CASE WHEN 会员卡距首曝天数 = 0 AND card_type = '飞享' THEN sign_amt ELSE 0 END)                   AS 飞享签约金额_T0
       ,SUM(CASE WHEN 会员卡距首曝天数 = 0 AND card_type = '飞跃' THEN sign_amt ELSE 0 END)                   AS 飞跃签约金额_T0
       ,SUM(CASE WHEN 会员卡距首曝天数 = 0 AND card_type = '提额' THEN sign_amt ELSE 0 END)                   AS 提额签约金额_T0
       ,SUM(CASE WHEN 会员卡距首曝天数 = 0 AND card_type = '飞享' THEN equity_income ELSE 0 END)              AS 飞享权益收入_T0
       ,SUM(CASE WHEN 会员卡距首曝天数 = 0 AND card_type = '飞跃' THEN equity_income ELSE 0 END)              AS 飞跃权益收入_T0
       ,SUM(CASE WHEN 会员卡距首曝天数 = 0 AND card_type = '提额' THEN equity_income ELSE 0 END)              AS 提额权益收入_T0
       ,SUM(CASE WHEN 会员卡距首曝天数 = 0 THEN equity_income ELSE 0 END)                                     AS 总权益收入_T0

       ,SUM(CASE WHEN 会员卡距首曝天数 BETWEEN 0 AND 6 AND card_type = '飞享' THEN sign_amt ELSE 0 END)       AS 飞享签约金额_T7
       ,SUM(CASE WHEN 会员卡距首曝天数 BETWEEN 0 AND 6 AND card_type = '飞跃' THEN sign_amt ELSE 0 END)       AS 飞跃签约金额_T7
       ,SUM(CASE WHEN 会员卡距首曝天数 BETWEEN 0 AND 6 AND card_type = '提额' THEN sign_amt ELSE 0 END)       AS 提额签约金额_T7
       ,SUM(CASE WHEN 会员卡距首曝天数 BETWEEN 0 AND 6 AND card_type = '飞享' THEN equity_income ELSE 0 END)  AS 飞享权益收入_T7
       ,SUM(CASE WHEN 会员卡距首曝天数 BETWEEN 0 AND 6 AND card_type = '飞跃' THEN equity_income ELSE 0 END)  AS 飞跃权益收入_T7
       ,SUM(CASE WHEN 会员卡距首曝天数 BETWEEN 0 AND 6 AND card_type = '提额' THEN equity_income ELSE 0 END)  AS 提额权益收入_T7
       ,SUM(CASE WHEN 会员卡距首曝天数 BETWEEN 0 AND 6 THEN equity_income ELSE 0 END)                         AS 总权益收入_T7

       ,SUM(CASE WHEN 会员卡距首曝天数 BETWEEN 0 AND 30 AND card_type = '飞享' THEN sign_amt ELSE 0 END)      AS 飞享签约金额_T30
       ,SUM(CASE WHEN 会员卡距首曝天数 BETWEEN 0 AND 30 AND card_type = '飞跃' THEN sign_amt ELSE 0 END)      AS 飞跃签约金额_T30
       ,SUM(CASE WHEN 会员卡距首曝天数 BETWEEN 0 AND 30 AND card_type = '提额' THEN sign_amt ELSE 0 END)      AS 提额签约金额_T30
       ,SUM(CASE WHEN 会员卡距首曝天数 BETWEEN 0 AND 30 AND card_type = '飞享' THEN equity_income ELSE 0 END) AS 飞享权益收入_T30
       ,SUM(CASE WHEN 会员卡距首曝天数 BETWEEN 0 AND 30 AND card_type = '飞跃' THEN equity_income ELSE 0 END) AS 飞跃权益收入_T30
       ,SUM(CASE WHEN 会员卡距首曝天数 BETWEEN 0 AND 30 AND card_type = '提额' THEN equity_income ELSE 0 END) AS 提额权益收入_T30
       ,SUM(CASE WHEN 会员卡距首曝天数 BETWEEN 0 AND 30 THEN equity_income ELSE 0 END)                        AS 总权益收入_T30
FROM vip_order_detail
GROUP BY  曝光日期
         ,group_tag
         ,risk_I20
         ,可用额度4k_10k
)

SELECT  l.*
       ,NVL(v.飞享签约金额_T0,0)  AS 飞享签约金额_T0
       ,NVL(v.飞跃签约金额_T0,0)  AS 飞跃签约金额_T0
       ,NVL(v.提额签约金额_T0,0)  AS 提额签约金额_T0
       ,NVL(v.飞享权益收入_T0,0)  AS 飞享权益收入_T0
       ,NVL(v.飞跃权益收入_T0,0)  AS 飞跃权益收入_T0
       ,NVL(v.提额权益收入_T0,0)  AS 提额权益收入_T0
       ,NVL(v.总权益收入_T0,0)    AS 总权益收入_T0
       ,NVL(v.飞享签约金额_T7,0)  AS 飞享签约金额_T7
       ,NVL(v.飞跃签约金额_T7,0)  AS 飞跃签约金额_T7
       ,NVL(v.提额签约金额_T7,0)  AS 提额签约金额_T7
       ,NVL(v.飞享权益收入_T7,0)  AS 飞享权益收入_T7
       ,NVL(v.飞跃权益收入_T7,0)  AS 飞跃权益收入_T7
       ,NVL(v.提额权益收入_T7,0)  AS 提额权益收入_T7
       ,NVL(v.总权益收入_T7,0)    AS 总权益收入_T7
       ,NVL(v.飞享签约金额_T30,0) AS 飞享签约金额_T30
       ,NVL(v.飞跃签约金额_T30,0) AS 飞跃签约金额_T30
       ,NVL(v.提额签约金额_T30,0) AS 提额签约金额_T30
       ,NVL(v.飞享权益收入_T30,0) AS 飞享权益收入_T30
       ,NVL(v.飞跃权益收入_T30,0) AS 飞跃权益收入_T30
       ,NVL(v.提额权益收入_T30,0) AS 提额权益收入_T30
       ,NVL(v.总权益收入_T30,0)   AS 总权益收入_T30
FROM loan_agg l
LEFT JOIN vip_agg v
ON l.曝光日期 = v.曝光日期 AND l.group_tag = v.group_tag AND l.可用额度4k_10k = v.可用额度4k_10k AND l.risk_I20 = v.risk_I20
'''

In [2]:
import query_analysis_tool as qat
loan_stats = qat.run_query(query)
file_path = r"D:\4.临时取数\20+权益资产测试\0519\消金20接资金路由0519后新逻辑_0605.xlsx"

# 字符串/分组字段
str_cols = ['曝光日期', 'group_tag', 'risk_I20', '可用额度4k_10k']

windows = ['T0', 'T7', 'T30']

# 浮点数字段
loan_float_metrics = ['放款金额', '定价', '期限', 'principal', 'annualized_interest_fee']
vip_cards = ['飞享', '飞跃', '提额']
vip_float_metrics = ['签约金额', '权益收入']

float_cols = [f'{metric}_{window}' for window in windows for metric in loan_float_metrics]
float_cols += [f'{card}{metric}_{window}' for window in windows for card in vip_cards for metric in vip_float_metrics]
float_cols += [f'总权益收入_{window}' for window in windows]

# 整数字段
int_cols = ['曝光人数'] + [f'放款笔数_{window}' for window in windows]

loan_stats = qat.format_dataframe_columns(
    loan_stats,
    str_cols=str_cols,
    date_cols=[],
    int_cols=int_cols,
    float_cols=float_cols
)

qat.write_dataframe_to_excel(
    file_path=file_path,
    dataframes_dict={"会员卡收入": loan_stats},
    start_row=1,
    include_header=True
)

正在获取数据，首段 SQL: 
WITH exposure_base AS
(
       SELECT  user_no
   ...
成功写入工作表: 会员卡收入
文件已保存: D:\4.临时取数\20+权益资产测试\0519\消金20接资金路由0519后新逻辑_0605.xlsx


In [ ]:
calc_fields = {}

for window in windows:
    calc_fields[f'{window}放款件均'] = {
        'formula': f"='放款金额_{window}'/'放款笔数_{window}'",
        'number_format': '#,##0'
    }
    calc_fields[f'{window}加权期限'] = {
        'formula': f"='期限_{window}'/'放款金额_{window}'",
        'number_format': '0.00'
    }
    calc_fields[f'{window}加权定价'] = {
        'formula': f"='定价_{window}'/'放款金额_{window}'",
        'number_format': '0.00%'
    }
    calc_fields[f'{window}息费率'] = {
        'formula': f"='annualized_interest_fee_{window}'/'principal_{window}'",
        'number_format': '0.00%'
    }
    calc_fields[f'{window}权益收入占放款'] = {
        'formula': f"='总权益收入_{window}'/'放款金额_{window}'",
        'number_format': '0.00%'
    }

    for card in vip_cards:
        display_card = '提额卡' if card == '提额' else card
        calc_fields[f'{window}{display_card}签约金额'] = {
            'formula': f"='{card}签约金额_{window}'/'放款金额_{window}'",
            'number_format': '0.00%'
        }
        calc_fields[f'{window}{display_card}权益收入'] = {
            'formula': f"='{card}权益收入_{window}'/'放款金额_{window}'",
            'number_format': '0.00%'
        }

qat.add_pivot_calculated_fields(
    file_path=file_path,
    sheet_name="summary",
    pivot_name="数据透视表16",
    fields=calc_fields
)

[Pivot] sheet=summary pivot=数据透视表16 cache_index=4
[OK] Add CalculatedField: T0放款件均 | ='放款金额_T0'/'放款笔数_T0'
[OK] Add to Values: T0放款件均 | format=#,##0.00
[OK] Add CalculatedField: T0加权期限 | ='期限_T0'/'放款金额_T0'
[OK] Add to Values: T0加权期限 | format=0.00
[OK] Add CalculatedField: T0加权定价 | ='定价_T0'/'放款金额_T0'
[OK] Add to Values: T0加权定价 | format=0.00%
[OK] Add CalculatedField: T0息费率 | ='annualized_interest_fee_T0'/'principal_T0'
[OK] Add to Values: T0息费率 | format=0.00%
[OK] Add CalculatedField: T0权益收入占放款 | ='总权益收入_T0'/'放款金额_T0'
[OK] Add to Values: T0权益收入占放款 | format=0.00%
[OK] Add CalculatedField: T0飞享签约金额 | ='飞享签约金额_T0'/'放款金额_T0'
[OK] Add to Values: T0飞享签约金额 | format=0.00%
[OK] Add CalculatedField: T0飞享权益收入 | ='飞享权益收入_T0'/'放款金额_T0'
[OK] Add to Values: T0飞享权益收入 | format=0.00%
[OK] Add CalculatedField: T0飞跃签约金额 | ='飞跃签约金额_T0'/'放款金额_T0'
[OK] Add to Values: T0飞跃签约金额 | format=0.00%
[OK] Add CalculatedField: T0飞跃权益收入 | ='飞跃权益收入_T0'/'放款金额_T0'
[OK] Add to Values: T0飞跃权益收入 | format=0.00%
[OK] Add Calculat

[]

#### 人维度

In [5]:
query = '''
SELECT  曝光日期
       ,group_tag
       ,飞跃在会
       ,飞享在会
       -- 人维度 
       ,COUNT(DISTINCT user_no)                                             AS 曝光人数
       ,COUNT(DISTINCT CASE WHEN order_number IS NOT NULL THEN user_no END) AS 提现人数
       ,COUNT(DISTINCT CASE WHEN loan_time IS NOT NULL THEN user_no END)    AS 放款人数

       ,COUNT(DISTINCT CASE WHEN 可用额度4k_10k = 1 THEN user_no END)        AS 可用额度4k_10k人数
       ,COUNT(DISTINCT CASE WHEN risk_I20 = 1       THEN user_no END)       AS 推荐过I20人数
       ,COUNT(DISTINCT CASE WHEN 风险原始定价 = 'I24' THEN user_no END)      AS 原始定价I24人数
       ,COUNT(DISTINCT CASE WHEN 风险原始定价 = 'I36' THEN user_no END)      AS 原始定价I36人数
 
       ,COUNT(DISTINCT CASE WHEN fy_sub_vip_type = 'fei_yue_i24' THEN user_no END)      AS 飞跃I24在会人数
       ,COUNT(DISTINCT CASE WHEN fy_sub_vip_type = 'fei_yue_i20' THEN user_no END)      AS 飞跃I20在会人数
       ,COUNT(DISTINCT CASE WHEN 飞跃在会状态 = '签约当笔' AND fy_sub_vip_type = 'fei_yue_i24' THEN user_no END)     AS 飞跃I24签约当笔人数
       ,COUNT(DISTINCT CASE WHEN 飞跃在会状态 = '签约当笔' AND fy_sub_vip_type = 'fei_yue_i20' THEN user_no END)     AS 飞跃I20签约当笔人数
       ,COUNT(DISTINCT CASE WHEN 飞享在会 = '在会' THEN user_no END)                     AS 飞享在会人数

FROM  xyf_jingying_dev.price20_preroute_cust_0519_lss_info
GROUP BY  曝光日期
         ,group_tag
         ,飞跃在会
         ,飞享在会
'''

In [6]:
import query_analysis_tool as qat
loan_stats = qat.run_query(query)
file_path = r"D:\4.临时取数\20+权益资产测试\0519\消金20接资金路由.xlsx"

# 字符串字段
str_cols = ['曝光日期','group_tag','飞跃在会','飞享在会']

# 浮点数字段 - 金额、加权值、占比类字段
float_cols = [
]

# 整数字段
int_cols = [
    '曝光人数','提现人数','放款人数','可用额度4k_10k人数','推荐过I20人数','原始定价I24人数','原始定价I36人数',
    '飞跃I24在会人数','飞跃I20在会人数','飞享在会人数',
    '飞跃I24签约当笔人数','飞跃I20签约当笔人数',
]

loan_stats = qat.format_dataframe_columns(
    loan_stats,
    str_cols=str_cols,
    date_cols=[],
    int_cols=int_cols,
    float_cols=float_cols
)

qat.write_dataframe_to_excel(
    file_path=file_path,
    dataframes_dict={"日曝光数据人维度": loan_stats},
    start_row=1,
    include_header=True
)


正在获取数据，首段 SQL: 
SELECT  曝光日期
       ,group_tag
       ,飞跃在会
      ...
成功写入工作表: 日曝光数据人维度
文件已保存: D:\4.临时取数\20+权益资产测试\0519\消金20接资金路由.xlsx


In [4]:
import query_analysis_tool as qat

# 新 SQL 已经直接产出发起率、放款率、期限、定价等字段；这里仅保留透视表里可能需要重复添加的 T0 计算字段。
calc_fields = {
    "T0发起率": {"formula": "='提现人数'/曝光人数", "number_format": "0.00%"},
    "T0放款率": {"formula": "='放款人数'/曝光人数", "number_format": "0.00%"},
   
    "可用额度4k_10k占比": {"formula": "='可用额度4k_10k人数'/曝光人数", "number_format": "0.00%"},
    "推荐过I20占比": {"formula": "='推荐过I20人数'/曝光人数", "number_format": "0.00%"},
    "原始定价I24占比": {"formula": "='原始定价I24人数'/曝光人数", "number_format": "0.00%"},
    "原始定价I36占比": {"formula": "='原始定价I36人数'/曝光人数", "number_format": "0.00%"},
    "飞跃I24在会占比": {"formula": "='飞跃I24在会人数'/曝光人数", "number_format": "0.00%"},
    "飞跃I20在会占比": {"formula": "='飞跃I20在会人数'/曝光人数", "number_format": "0.00%"},
    
    "飞享在会占比": {"formula": "='飞享在会人数'/曝光人数", "number_format": "0.00%"},
    "飞跃I24签约当笔占比": {"formula": "='飞跃I24签约当笔人数'/曝光人数", "number_format": "0.00%"},
    "飞跃I20签约当笔占比": {"formula": "='飞跃I20签约当笔人数'/曝光人数", "number_format": "0.00%"},
}

qat.add_pivot_calculated_fields(
    file_path=file_path,
    sheet_name="summary",
    pivot_name="数据透视表12",
    fields=calc_fields
)


[Pivot] sheet=summary pivot=数据透视表12 cache_index=1
[Skip] T0发起率 已存在，跳过
[Skip] T0放款率 已存在，跳过
[Skip] 可用额度4k_10k占比 已存在，跳过
[OK] Add CalculatedField: 推荐过I20占比 | ='推荐过I20人数'/曝光人数
[OK] Add to Values: 推荐过I20占比 | format=0.00%
[Skip] 原始定价I24占比 已存在，跳过
[Skip] 原始定价I36占比 已存在，跳过
[Skip] 飞跃I24在会占比 已存在，跳过
[Skip] 飞跃I20在会占比 已存在，跳过
[Skip] 飞享在会占比 已存在，跳过
[Skip] 飞跃I24签约当笔占比 已存在，跳过
[Skip] 飞跃I20签约当笔占比 已存在，跳过

全部计算字段设置成功。



[]

#### 看订单

In [13]:
query = '''
SELECT  曝光日期
       ,group_tag
       ,飞跃在会
       ,飞享在会
       ,fy_sub_vip_type
       ,asset_type_flag
       ,可用额度4k_10k
       ,风险原始定价
       ,飞跃在会状态
       
       ,COUNT(DISTINCT user_no)                                             AS 曝光人数
       ,COUNT(DISTINCT CASE WHEN order_number IS NOT NULL THEN user_no END) AS 提现人数
       -- 订单维度 
       ,COUNT(DISTINCT order_number)                                                                                                  AS 发起订单
       ,COUNT(DISTINCT CASE WHEN (UNIX_TIMESTAMP(risk_success_time) - UNIX_TIMESTAMP(first_order_time)) <= 2 * 3600 THEN order_number END)  AS 风险通过_2h
       ,COUNT(DISTINCT CASE WHEN (UNIX_TIMESTAMP(risk_success_time) - UNIX_TIMESTAMP(first_order_time)) <= 24 * 3600 THEN order_number END) AS 风险通过_24h
       ,COUNT(DISTINCT CASE WHEN (UNIX_TIMESTAMP(risk_success_time) - UNIX_TIMESTAMP(first_order_time)) <= 72 * 3600 THEN order_number END) AS 风险通过_72h
       ,COUNT(DISTINCT CASE WHEN risk_success_time IS NOT NULL THEN order_number END)                                                       AS 风险通过
       ,COUNT(DISTINCT CASE WHEN (UNIX_TIMESTAMP(loan_time) - UNIX_TIMESTAMP(risk_success_time)) <= 2 * 3600 THEN order_number END)         AS 资金通过_2h
       ,COUNT(DISTINCT CASE WHEN (UNIX_TIMESTAMP(loan_time) - UNIX_TIMESTAMP(risk_success_time)) <= 24 * 3600 THEN order_number END)        AS 资金通过_24h
       ,COUNT(DISTINCT CASE WHEN (UNIX_TIMESTAMP(loan_time) - UNIX_TIMESTAMP(risk_success_time)) <= 72 * 3600 THEN order_number END)        AS 资金通过_72h
       ,COUNT(DISTINCT CASE WHEN loan_time IS NOT NULL THEN order_number END)                                                               AS 放款通过

       ,SUM(CASE WHEN risk_status = 'pass' THEN order_amt ELSE 0 END)                                                            AS 资产
       ,SUM(CASE WHEN loan_time IS NOT NULL THEN loan_amt ELSE 0 END)                                                            AS 放款金额
       ,SUM(CASE WHEN loan_time IS NOT NULL THEN loan_amt * period ELSE 0 END)                                                   AS 期限
       ,SUM(CASE WHEN loan_time IS NOT NULL THEN loan_amt * fee_rate ELSE 0 END)                                                 AS 定价

FROM  xyf_jingying_dev.price20_preroute_cust_0519_lss_info
GROUP BY  曝光日期
         ,group_tag
         ,飞跃在会
         ,飞享在会
         ,fy_sub_vip_type
         ,asset_type_flag
         ,可用额度4k_10k
         ,风险原始定价
         ,飞跃在会状态
'''

In [14]:
import query_analysis_tool as qat
loan_stats = qat.run_query(query)
file_path = r"D:\4.临时取数\20+权益资产测试\0519\消金20接资金路由.xlsx"

# 字符串字段
str_cols = ['曝光日期','风险原始定价','group_tag','飞跃在会','飞享在会','fy_sub_vip_type','可用额度4k_10k','asset_type_flag']

# 浮点数字段 - 金额、加权值、占比类字段
float_cols = [ '资产','放款金额','期限','定价']

# 整数字段
int_cols = [
    '曝光人数','提现人数','发起订单','风险通过_2h','风险通过_24h','风险通过_72h', '风险通过','资金通过_2h','资金通过_24h','资金通过_72h','放款通过'
]

loan_stats = qat.format_dataframe_columns(
    loan_stats,
    str_cols=str_cols,
    date_cols=[],
    int_cols=int_cols,
    float_cols=float_cols
)

qat.write_dataframe_to_excel(
    file_path=file_path,
    dataframes_dict={"日曝光数据订单维度": loan_stats},
    start_row=1,
    include_header=True
)


正在获取数据，首段 SQL: 
SELECT  曝光日期
       ,group_tag
       ,飞跃在会
      ...
成功写入工作表: 日曝光数据订单维度
文件已保存: D:\4.临时取数\20+权益资产测试\0519\消金20接资金路由.xlsx


In [12]:
import query_analysis_tool as qat

calc_fields = {
    "T0发起率": {"formula": "='提现人数'/曝光人数", "number_format": "0.00%"},
    
    "T0放款件均": {"formula": "='放款金额'/放款通过", "number_format": "0.00"},
    "T0加权期限": {"formula": "='期限'/放款金额", "number_format": "0.00"},
    "T0加权定价": {"formula": "='定价'/放款金额", "number_format": "0.00%"},

    "风险2h通过率": {"formula": "='风险通过_2h'/发起订单", "number_format": "0.00%"},
    "风险24h通过率": {"formula": "='风险通过_24h'/发起订单", "number_format": "0.00%"},
    "风险72h通过率": {"formula": "='风险通过_72h'/发起订单", "number_format": "0.00%"},
    "资金2h通过率": {"formula": "='资金通过_2h'/风险通过", "number_format": "0.00%"},
    "资金24h通过率": {"formula": "='资金通过_24h'/风险通过", "number_format": "0.00%"},
    "资金72h通过率": {"formula": "='资金通过_72h'/风险通过", "number_format": "0.00%"},
}

qat.add_pivot_calculated_fields(
    file_path=file_path,
    sheet_name="summary",
    pivot_name="数据透视表14",
    fields=calc_fields
)


[Pivot] sheet=summary pivot=数据透视表14 cache_index=2
[OK] Add CalculatedField: T0发起率 | ='提现人数'/曝光人数
[OK] Add to Values: T0发起率 | format=0.00%
[OK] Add CalculatedField: T0放款件均 | ='放款金额'/放款通过
[OK] Add to Values: T0放款件均 | format=0.00
[OK] Add CalculatedField: T0加权期限 | ='期限'/放款金额
[OK] Add to Values: T0加权期限 | format=0.00
[OK] Add CalculatedField: T0加权定价 | ='定价'/放款金额
[OK] Add to Values: T0加权定价 | format=0.00%
[OK] Add CalculatedField: 风险2h通过率 | ='风险通过_2h'/发起订单
[OK] Add to Values: 风险2h通过率 | format=0.00%
[OK] Add CalculatedField: 风险24h通过率 | ='风险通过_24h'/发起订单
[OK] Add to Values: 风险24h通过率 | format=0.00%
[OK] Add CalculatedField: 风险72h通过率 | ='风险通过_72h'/发起订单
[OK] Add to Values: 风险72h通过率 | format=0.00%
[OK] Add CalculatedField: 资金2h通过率 | ='资金通过_2h'/风险通过
[OK] Add to Values: 资金2h通过率 | format=0.00%
[OK] Add CalculatedField: 资金24h通过率 | ='资金通过_24h'/风险通过
[OK] Add to Values: 资金24h通过率 | format=0.00%
[OK] Add CalculatedField: 资金72h通过率 | ='资金通过_72h'/风险通过
[OK] Add to Values: 资金72h通过率 | format=0.00%

全部计算字段设置成功。



[]

#### 飞跃签约当笔会员卡收入

In [6]:
query = '''
-- by日：当天签约当笔会员卡收入
WITH loan_order_detail AS
(
	SELECT  *
	FROM xyf_jingying.price20_preroute_cust_0519_lss_info
	WHERE 飞跃在会状态 = '签约当笔'
	AND loan_time IS NOT NULL 
), repay_plan AS
(
	SELECT  order_number
	       ,SUM(initial_principal)                                                          AS principal
	       ,SUM(initial_interest) + SUM(initial_after_loan_fee) + SUM(initial_platform_fee) AS interest_fee
	FROM xyf_dwd.dwd_repay_loan_repay_plan_df
	WHERE pt = MAX_PT('xyf_dwd.dwd_repay_loan_repay_plan_df')
	GROUP BY  order_number
),
order_level AS
(
    SELECT  a.*
           ,b.principal
           ,b.interest_fee
           -- 飞跃：签约当笔
           ,NVL(a.fy_real_card_price,0) AS 飞跃签约金额
           ,CASE WHEN a.fy_pay_time IS NOT NULL THEN NVL(a.fy_real_card_price,0) ELSE 0 END
              - CASE WHEN a.fy_act_refund_time IS NOT NULL THEN NVL(a.fy_refund_amount,0) ELSE 0 END AS 飞跃权益收入
           -- 提额：底表 tek 已按 first_order_number 聚合
           ,NVL(a.tek_pay_amt,0) AS 提额签约金额
           ,NVL(a.tek_pay_amt,0) - NVL(a.tek_refund_amt,0) AS 提额权益收入
    FROM loan_order_detail a
    LEFT JOIN repay_plan b
    ON a.order_number = b.order_number
)
SELECT  曝光日期
       ,group_tag
       ,fy_sub_vip_type
       ,asset_type_flag
       ,可用额度4k_10k
       ,风险原始定价
       ,COUNT(DISTINCT user_no) AS 放款人数
       ,COUNT(DISTINCT order_number) AS 放款笔数
       ,SUM(loan_amt) AS 放款金额
       ,SUM(loan_amt * fee_rate) AS 定价
       ,SUM(loan_amt * period) AS 期限
       ,SUM(principal) AS principal
       ,SUM(interest_fee) AS annualized_interest_fee
       ,SUM(飞跃签约金额) AS 飞跃签约金额
       ,SUM(提额签约金额) AS 提额签约金额
       ,SUM(飞跃权益收入) AS 飞跃权益收入
       ,SUM(提额权益收入) AS 提额权益收入
       ,SUM(飞跃权益收入 + 提额权益收入) AS 总权益收入
FROM order_level
GROUP BY  曝光日期
         ,group_tag
         ,fy_sub_vip_type
         ,asset_type_flag
         ,可用额度4k_10k
         ,风险原始定价
'''

In [8]:
import query_analysis_tool as qat
loan_stats = qat.run_query(query)
file_path = r"D:\4.临时取数\20+权益资产测试\0519\消金20接资金路由0519后新逻辑_0605.xlsx"
# 字符串字段
str_cols = ['曝光日期','group_tag','fy_sub_vip_type','asset_type_flag','可用额度4k_10k','风险原始定价']


# 浮点数字段 - 保留2位小数，所有有应还本金的字段均为浮点数

float_cols = [ '放款金额', '定价', '期限', 'principal', 'annualized_interest_fee', 
             '飞享签约金额', '飞跃签约金额', '提额签约金额', '飞享权益收入', '飞跃权益收入', '提额权益收入', '总权益收入']

# 整数字段
int_cols = [ '放款人数', '放款笔数']

loan_stats = qat.format_dataframe_columns(
    loan_stats,
    str_cols=str_cols,
    date_cols=[], 
    int_cols=int_cols,
    float_cols=float_cols
)

qat.write_dataframe_to_excel(
    file_path=file_path,
    dataframes_dict={"签约当笔会员卡收入": loan_stats},
    start_row=1,
    include_header=True
)

正在获取数据，首段 SQL: 
-- by日：当天签约当笔会员卡收入
WITH loan_order_detail AS
(
	S ...
成功写入工作表: 签约当笔会员卡收入
文件已保存: D:\4.临时取数\20+权益资产测试\0519\消金20接资金路由0519后新逻辑_0605.xlsx


In [6]:
import query_analysis_tool as qat
qat.clear_pivot_calculated_fields(
    file_path=file_path,
    sheet_name="summary",
    pivot_name="数据透视表15",
)


calc_fields = {
    "放款件均": {"formula": "='放款金额'/'放款笔数'", "number_format": "#,##0"},
    "加权期限": {"formula": "='期限'/'放款金额'", "number_format": "0.00"},
    "加权定价": {"formula": "='定价'/'放款金额'", "number_format": "0.00%"},

    "息费率": {"formula": "='annualized_interest_fee'/'principal'", "number_format": "0.00%"},
    "权益收入占放款": {"formula": "='总权益收入'/'放款金额'", "number_format": "0.00%"},
    "飞享卡签约金额": {"formula": "='飞享签约金额'/'放款金额'", "number_format": "0.00%"},
    "飞享卡权益收入": {"formula": "='飞享权益收入'/'放款金额'", "number_format": "0.00%"},
    "飞跃卡签约金额": {"formula": "='飞跃签约金额'/'放款金额'", "number_format": "0.00%"},
    "飞跃卡权益收入": {"formula": "='飞跃权益收入'/'放款金额'", "number_format": "0.00%"},
    "提额卡签约金额": {"formula": "='提额签约金额'/'放款金额'", "number_format": "0.00%"},
    "提额卡权益收入": {"formula": "='提额权益收入'/'放款金额'", "number_format": "0.00%"},
}

qat.add_pivot_calculated_fields(
    file_path=file_path,
    sheet_name="summary",
    pivot_name="数据透视表15",
    fields=calc_fields
) 

[Pivot] sheet=summary pivot=数据透视表15
[OK] Deleted CalculatedField: 放款件均
[OK] Deleted CalculatedField: 加权期限
[OK] Deleted CalculatedField: 加权定价
[OK] Deleted CalculatedField: 息费率
[OK] Deleted CalculatedField: 权益收入占放款
[OK] Deleted CalculatedField: 提额卡签约金额
[OK] Deleted CalculatedField: 提额卡权益收入
[OK] Deleted CalculatedField: 飞享卡签约金额
[OK] Deleted CalculatedField: 飞享卡权益收入
[OK] Deleted CalculatedField: 飞跃卡签约金额
[OK] Deleted CalculatedField: 飞跃卡权益收入
[OK] Deleted CalculatedField: 当天放款件均

计算字段清空完成。

[Pivot] sheet=summary pivot=数据透视表15 cache_index=3
[OK] Add CalculatedField: 放款件均 | ='放款金额'/'放款笔数'
[OK] Add to Values: 放款件均 | format=#,##0
[OK] Add CalculatedField: 加权期限 | ='期限'/'放款金额'
[OK] Add to Values: 加权期限 | format=0.00
[OK] Add CalculatedField: 加权定价 | ='定价'/'放款金额'
[OK] Add to Values: 加权定价 | format=0.00%
[OK] Add CalculatedField: 息费率 | ='annualized_interest_fee'/'principal'
[OK] Add to Values: 息费率 | format=0.00%
[OK] Add CalculatedField: 权益收入占放款 | ='总权益收入'/'放款金额'
[OK] Add to Values: 权益收入占放款 | format=0.

[]